<div align="center">
  <a href="https://colab.research.google.com/github/ActionPace/ensemble-divergence/blob/main/ensemble_divergence_social_science.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

# Ensemble Divergence — Methods Description Analysis
### *Toward a Control Architecture for Composable AI Execution Systems*
**David E. King · ActionPace · April 2026**

---

This notebook extends the ensemble divergence methodology from source code analysis
to **empirical research methods descriptions** — identifying underspecification zones
before any re-implementation attempt is made.

**Key demonstration:** Applied to Settele (2022), a documented failure case from
Kohler et al. (2026) *"Read the Paper, Write the Code"*, both models independently
flag the party identification coding as the primary ambiguity zone — the exact
failure node that caused reproduction divergence in that study.

The ensemble pairs **Apertus** (Swiss AI, Llama-family) with **Mistral** (French,
independent architecture). Where both independently locate the same underspecified
contract, the finding is robust across training lineages.

### Notebook structure

| Section | Runtime | What it shows |
|---------|---------|---------------|
| **Section 1** | T4 free tier | Apertus 8B vs Mistral 7B on Settele (2022) methods |
| **Section 2** | A100 (any paid plan) | Apertus 70B vs Mistral 7B — depth-of-read |
| **Section 3** | A100 (any paid plan) | Apertus 70B vs Mistral Nemo 12B — full ensemble |

If using A100, choose "High-RAM" to allow max model layers to be used in the GPU.

---
**Repository:** [github.com/ActionPace/ensemble-divergence](https://github.com/ActionPace/ensemble-divergence)  
**Paper:** arXiv forthcoming · cs.SE / cs.AI  
**Reference:** Kohler, Zollikofer, Einsiedler, Hoyle & Ash (2026) — arXiv:2604.21965


In [ ]:
#@title Detect hardware and set runtime flags
import subprocess, shutil, psutil, os

machine_type        = "CPU"
total_gpu_memory_gb = 0.0
total_ram_gb        = psutil.virtual_memory().total / 1024**3
_, _, free_disk     = shutil.disk_usage("/")
free_disk_gb        = free_disk / 1024**3

try:
    gpu_name = subprocess.check_output(
        "nvidia-smi --query-gpu=gpu_name --format=csv,noheader",
        shell=True, text=True).strip()
    smi_out = subprocess.check_output(
        "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits",
        shell=True, text=True).strip()
    total_gpu_memory_gb = int(smi_out.splitlines()[0]) / 1024
    if   "T4"   in gpu_name: machine_type = "T4"
    elif "L4"   in gpu_name: machine_type = "L4"
    elif "A100" in gpu_name: machine_type = "A100"
    else:                    machine_type = "GPU_Other"
except Exception:
    gpu_name = "None"

is_a100       = machine_type == "A100"
is_high_ram   = total_gpu_memory_gb >= 75

can_run_s1 = True
can_run_s2 = is_a100
can_run_s3 = is_a100 and is_high_ram

# ── A100 partial-layer config ──────────────────────────────────────────────────
# 70B Q5_K_S is ~47 GB. Standard A100 = 40 GB VRAM.
# Set to number of transformer layers to keep on GPU; rest offload to CPU RAM.
# Start at 60 and reduce by 4 until it loads without OOM.
# Leave as -1 if you have 80GB A100 or High-RAM runtime enabled.
A100_70B_PARTIAL_LAYERS = 72   # <- fill in after testing

if is_a100 and is_high_ram:
    layers_70b = -1
elif A100_70B_PARTIAL_LAYERS != -1:
    layers_70b = A100_70B_PARTIAL_LAYERS
else:
    layers_70b = -1

print("=" * 60)
print(" RUNTIME SUMMARY")
print("=" * 60)
print(f"  GPU:          {gpu_name}")
print(f"  GPU VRAM:     {total_gpu_memory_gb:.1f} GiB")
print(f"  System RAM:   {total_ram_gb:.1f} GiB")
print(f"  Free disk:    {free_disk_gb:.1f} GiB")
print()
print("=" * 60)
print(" SECTION AVAILABILITY")
print("=" * 60)
print(f"  Section 1 (Apertus 8B vs Mistral 7B):        {'YES' if can_run_s1 else 'needs GPU'}")
print(f"  Section 2 (Apertus 70B vs Mistral 7B):       {'YES' if can_run_s2 else 'needs A100 — any paid Colab plan'}")
print(f"  Section 3 (Apertus 70B vs Mistral Nemo 12B): {'YES' if can_run_s2 else 'needs A100 — any paid Colab plan'}")
if is_a100 and not is_high_ram and A100_70B_PARTIAL_LAYERS == -1:
    print()
    print("  NOTE: A100 40GB detected. Section 2 will attempt full 70B load.")
    print("  If OOM: set A100_70B_PARTIAL_LAYERS above (start at 60, step -4)")
    print("  Or: Runtime > Change runtime type > High-RAM (Pro only)")
print("=" * 60)

 RUNTIME SUMMARY
  GPU:          NVIDIA A100-SXM4-80GB
  GPU VRAM:     80.0 GiB
  System RAM:   167.1 GiB
  Free disk:    192.6 GiB

 SECTION AVAILABILITY
  Section 1 (Apertus 8B vs Mistral 7B):        YES
  Section 2 (Apertus 70B vs Mistral 7B):       YES
  Section 3 (Apertus 70B vs Mistral Nemo 12B): YES


---
## Section 1 — Free Demo: Apertus 8B vs Mistral 7B
**Runs on T4 free tier. No credentials required.**

Two 8B-class models from different countries and architectures analyze the same file.
Where they agree: high-confidence structural finding.
Where they disagree: a composition seam.

**Target:** `libs/langgraph/langgraph/pregel/main.py` — core execution loop of LangGraph,
one of the six frameworks surveyed in the paper. Identified as high-opacity with
load-bearing implicit contracts in the original survey.


In [ ]:
#@title 1.1 Install dependencies and download llama.cpp
from huggingface_hub import hf_hub_download

llama_file = f"llama.cpp-{machine_type}.tar.gz"
print(f"Downloading llama.cpp for {machine_type}...")
hf_hub_download(repo_id="actionpace/Llama-Cpp-Compiles",
                filename=llama_file, local_dir="/content")
if os.path.isdir("/content/llama.cpp"):
    os.system("rm -rf /content/llama.cpp")
os.system(f"tar xf /content/{llama_file} -C /content/")
print("llama.cpp ready")

llama.cpp-A100.tar.gz:   0%|          | 0.00/338M [00:00<?, ?B/s]

llama.cpp ready


In [ ]:
#@title 1.2 Download Apertus 8B and Mistral 7B
APERTUS_8B_FILE = "Apertus-8B-Instruct-2509-Q5_K_M.gguf"
APERTUS_8B_REPO = "unsloth/Apertus-8B-Instruct-2509-GGUF"

MISTRAL_7B_FILE = "Mistral-7B-Instruct-v0.3-Q5_K_M.gguf"
MISTRAL_7B_REPO = "bartowski/Mistral-7B-Instruct-v0.3-GGUF"

for repo, fname in [(APERTUS_8B_REPO, APERTUS_8B_FILE),
                    (MISTRAL_7B_REPO,  MISTRAL_7B_FILE)]:
    if not os.path.exists(f"/content/{fname}"):
        print(f"Downloading {fname}...")
        hf_hub_download(repo_id=repo, filename=fname, local_dir="/content")
    print(f"  {fname}")

Apertus-8B-Instruct-2509-Q5_K_M.gguf:   0%|          | 0.00/5.81G [00:00<?, ?B/s]

  Apertus-8B-Instruct-2509-Q5_K_M.gguf


Mistral-7B-Instruct-v0.3-Q5_K_M.gguf:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

  Mistral-7B-Instruct-v0.3-Q5_K_M.gguf


In [ ]:
#@title 1.4 Define prompts, helpers, and corpus
import json, time
from openai import OpenAI
import json as _json

METHODS_OPACITY_SYSTEM = """You are a methodological specification analyst rating the
underspecification of an empirical research methods description.

UNDERSPECIFICATION = the gap between what a methods description claims to specify
and what a re-implementer actually needs to reproduce the analysis. This is about
implicit contracts: unstated variable mappings, undeclared software defaults,
missing data filtering rules, assumed normalization choices.

Score each dimension 0-20 and sum for total (0-100):

1. UNDECLARED_PRECONDITIONS (0-20)
   How many things must the re-implementer assume or infer that are stated
   nowhere in the methods description? (variable codings, data filters,
   software settings that must be guessed)

2. SOFTWARE_DEFAULT_RISK (0-20)
   How much does the analysis depend on software-specific defaults — estimator
   variants, F-statistic types, standard error flavors — that are not declared?
   High scores mean results will differ across software environments unless the
   implementer guesses the right defaults.

3. IMPLICIT_DATA_COUPLING (0-20)
   How many dependencies exist between the methods description and the raw data
   structure (column names, coding conventions, categorical mappings) that are
   not declared in the text? High scores mean the implementer must read the
   data to understand the methods.

4. CONTRACT_COMPLETENESS (0-20, inverted — 0 = fully specified)
   How incomplete are the analytical contracts? A fully specified methods section
   would allow exact re-implementation without access to the original code.
   Score 0 if all choices are declared; score 20 if almost nothing is declared.

5. RESULT_CENTRALITY (0-20)
   How load-bearing are the underspecified elements? High scores mean that
   guessing wrong on an undeclared choice will change the sign or significance
   of a primary result — not just a robustness check.

Return ONLY valid JSON, no preamble, no markdown fences:
{
  "undeclared_preconditions": <int 0-20>,
  "software_default_risk": <int 0-20>,
  "implicit_data_coupling": <int 0-20>,
  "contract_completeness": <int 0-20>,
  "result_centrality": <int 0-20>,
  "total": <int 0-100>,
  "opacity_score": <float 0.0-1.0>,
  "primary_signal": "<one sentence: the most important undeclared contract>",
  "ambiguity_zones": ["<zone1>", "<zone2>", "<zone3>"],
  "summary": "<2 sentences: specification assessment>"
}"""

IMPLEMENTER_BURDEN_SYSTEM = """You are analyzing an empirical research methods description
to identify what a re-implementer must know or assume that is not written anywhere.

Return ONLY valid JSON, no preamble, no markdown fences:
{
  "must_know": [
    {"item": "<what the implementer must know>", "stated_in_methods": <true|false>},
    {"item": "<what the implementer must know>", "stated_in_methods": <true|false>},
    {"item": "<what the implementer must know>", "stated_in_methods": <true|false>}
  ],
  "riskiest_assumption": "<the assumption most likely to produce wrong results if guessed incorrectly>",
  "silent_failure_mode": "<what incorrect reproduction would look like — results that seem plausible but are wrong>"
}"""


# ── SETTELE (2022) METHODS EXCERPT ────────────────────────────────────────────
#
# Source: "How Do Beliefs about the Gender Wage Gap Affect the Demand for
# Public Policy?" — Sonja Settele, AEJ: Economic Policy (2022)
#
# This excerpt covers the core regression specification for Table 2,
# which is the documented failure case in Kohler et al. (2026):
# agents must infer that survey codes A4 and A5 indicate Democrat support.
# The methods description does not declare this mapping.

SETTELE_METHODS = """
Study: How Do Beliefs about the Gender Wage Gap Affect the Demand for Public Policy?
Journal: American Economic Journal: Economic Policy (2022)
Author: Sonja Settele

SURVEY DESIGN
The study uses an online survey experiment with a nationally representative sample
of 4,021 U.S. respondents. Respondents were randomly assigned to one of four
information treatments about the gender wage gap, or a control group receiving
no information.

MAIN OUTCOME VARIABLE
The primary outcome is support for gender equality policies. Respondents indicated
their support for five policies on a scale from 1 (strongly oppose) to 7 (strongly
support): (1) equal pay legislation, (2) paid family leave, (3) childcare subsidies,
(4) affirmative action in hiring, and (5) gender quotas in corporate boards.
An index of policy support is constructed as the average of the five items.

TREATMENT ASSIGNMENT
Each respondent was randomly assigned to one of the following conditions:
- Control: No information provided
- Treatment 1: Informed that the gender wage gap is 18 cents per dollar
- Treatment 2: Informed that the gap is larger than they believed
- Treatment 3: Informed that the gap is smaller than they believed
- Treatment 4: Combination treatment

MAIN REGRESSION SPECIFICATION
The main estimating equation is:

  Y_i = alpha + beta * Treatment_i + gamma * X_i + epsilon_i

Where Y_i is the policy support index for respondent i, Treatment_i is an
indicator for assignment to the information treatment, and X_i is a vector
of individual-level controls including age, gender, income, and education.

HETEROGENEITY ANALYSIS
A key finding of the paper is that treatment effects differ substantially by
political affiliation. The paper reports results separately for Democrats and
Republicans. The analysis of partisan heterogeneity uses the self-reported
party identification variable from the survey.

The partisan gap in treatment effects is defined as the difference between
the treatment coefficient for Democrats and the treatment coefficient for
Republicans. Standard errors are clustered at the individual level.

DATA
The survey data is available in the replication package on OpenICPSR.
The party identification variable is coded using the standard ANES
five-point scale, where the scale captures strong Democrat through
strong Republican identification."""

# ── WHAT THE CODE ACTUALLY DOES (ground truth from Kohler et al.) ─────────────
#
# The original Stata code uses codes A4 and A5 to identify Democrats.
# The methods description above says "ANES five-point scale" but does not
# specify which numeric codes map to "Democrat" — that information exists
# only in the raw data codebook, which is not summarized in the paper.
#
# Kohler et al. (2026) Table 3 documents this as a verified failure case:
# one agent incorrectly assumed codes A1/A2 = Democrat (grade D)
# one agent correctly assumed codes A4/A5 = Democrat (grade B)
# The paper itself provides no basis for choosing between these interpretations.


def extract_corpus(path, max_lines=300):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()
    if len(lines) <= max_lines:
        return "".join(lines), len(lines)
    header = lines[:150]
    kws = ["if ","for ","while ","try:","except","def ","class ",
           "return ","raise ","async ","await ","yield "]
    best, bd = 150, 0
    for i in range(150, max(151, len(lines)-50)):
        d = sum(1 for l in lines[i:i+50] if any(k in l for k in kws))
        if d > bd: bd, best = d, i
    focal = lines[max(0,best-10):best+140]
    return (f"// HEADER ({len(header)} lines)\n" + "".join(header)
            + f"\n// FOCAL (line ~{best})\n" + "".join(focal))[:12000], len(lines)


def parse_json(raw):
    """Robustly extract JSON from model output.
    Handles: markdown fences, prose preamble, truncated output,
    opacity_score returned as 0-100 instead of 0-1.
    Returns empty dict on total failure rather than raising.
    """
    if not raw or not raw.strip():
        print("  [parse_json] Empty response")
        return {}
    raw = raw.strip()

    # Strip markdown fences if present
    if "```" in raw:
        for p in raw.split("```"):
            p = p.strip().lstrip("json").strip()
            if p.startswith("{"): raw = p; break

    # Find the first { anywhere in the response (handles prose preamble)
    first_brace = raw.find("{")
    if first_brace > 0:
        raw = raw[first_brace:]

    # Trim to last closing brace (handles truncated trailing text)
    last_brace = raw.rfind("}")
    if last_brace != -1:
        raw = raw[:last_brace + 1]

    # Fix Python literal booleans/None — 8B models often output these
    # instead of valid JSON true/false/null
    import re
    raw = re.sub(r'\bFalse\b', 'false', raw)
    raw = re.sub(r'\bTrue\b',  'true',  raw)
    raw = re.sub(r'\bNone\b',  'null',  raw)

    try:
        r = json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"  [parse_json] JSONDecodeError: {e}")
        print(f"  [parse_json] Raw (first 200 chars): {raw[:200]}")
        return {}

    # Normalize opacity_score if model returned 0-100 instead of 0-1
    if r.get("opacity_score", 0) > 1.0:
        r["opacity_score"] = r["opacity_score"] / 100.0
    return r


def start_server(model_file, n_gpu_layers=-1, wait=30):
    os.system("pkill -f llama-server 2>/dev/null")
    time.sleep(3)
    cmd = (f"/content/llama.cpp/build/bin/llama-server "
           f"-m /content/{model_file} -c 4096 -ngl {n_gpu_layers} "
           f"--port 12345 --log-file /content/server.log "
           f"--host 0.0.0.0 --jinja")
    subprocess.Popen(f"nohup {cmd} &", shell=True)
    print(f"  Loading {model_file} ({wait}s)...")
    time.sleep(wait)
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:12345/health", timeout=5)
        print("  Server ready")
    except Exception:
        print("  Still loading — proceeding")

client = OpenAI(api_key="dummy", base_url="http://localhost:12345/v1")

json_schema = """{
  "type": "object",
  "properties": {
    "undeclared_preconditions": {"type": "integer", "minimum": 0, "maximum": 20},
    "software_default_risk":        {"type": "integer", "minimum": 0, "maximum": 20},
    "implicit_data_coupling":        {"type": "integer", "minimum": 0, "maximum": 20},
    "contract_completeness":    {"type": "integer", "minimum": 0, "maximum": 20},
    "result_centrality":    {"type": "integer", "minimum": 0, "maximum": 20},
    "total":                    {"type": "integer", "minimum": 0, "maximum": 100},
    "opacity_score":            {"type": "number",  "minimum": 0, "maximum": 1},
    "primary_signal":           {"type": "string"},
    "ambiguity_zones":   {"type": "array", "items": {"type": "string"}},
    "summary":                  {"type": "string"}
  },
  "required": ["undeclared_preconditions","software_default_risk","implicit_data_coupling",
               "contract_completeness","result_centrality","total",
               "opacity_score","primary_signal","ambiguity_zones","summary"]
}"""


json_schema_dict = _json.loads(json_schema)

def run_prompt(system, label, max_tokens=800, retries=2):
    """Run a prompt with automatic retry on empty/invalid JSON.
    On retry: increases max_tokens and raises temperature slightly
    to encourage the model to complete its response.
    Returns empty dict if all retries fail.
    """
    for attempt in range(retries + 1):
        if attempt == 0:
            print(f"  Running {label}...")
        else:
            print(f"  Retrying {label} (attempt {attempt+1}, tokens={max_tokens+256*attempt})...")
        try:
            resp = client.chat.completions.create(
                model="local-model",
                max_tokens=max_tokens + 256 * attempt,
                temperature=0.1 + 0.05 * attempt,
                response_format={
                      "type": "json_schema",
                      "json_schema": {
                          "name": "opacity_result",
                          "strict": True,
                          "schema": json_schema_dict
                      }
                  },
                messages=[{"role": "system", "content": system},
                           {"role": "user",   "content": user_prompt}]
            )
            raw = resp.choices[0].message.content or ""
            result = parse_json(raw)
            print(result)
            if result:  # non-empty dict = success
                return result
            print(f"  Empty result on attempt {attempt+1}")
        except Exception as e:
            print(f"  API error on attempt {attempt+1}: {e}")
    print(f"  All retries failed for {label} — returning empty result")
    return {}


def print_result(r, label):
    if not r:
        print(f"  No result to display for {label} — model returned unparseable output")
        return
    t = r.get("total", 0)
    # Recalculate total from dimensions if model returned 0 total but non-zero dims
    if t == 0:
        dims_sum = sum(r.get(k, 0) for k in [
            "undeclared_preconditions", "global_state_risk",
            "implicit_coupling", "contract_completeness", "structural_centrality"
        ])
        if dims_sum > 0: t = dims_sum
    bar = "X"*int(t*32//100) + "."*(32-int(t*32//100))
    tier = "CRITICAL" if t>=80 else "HIGH" if t>=60 else "MEDIUM" if t>=40 else "LOW"
    # Determine display context — methods description or source file
    context = (f"Methods: {user_prompt[:60].split(chr(10))[1].strip()}"
               if user_prompt == SETTELE_METHODS
               else f"File: {TARGET_PATH}/{TARGET_FILE}")
    print(f"\n{'='*65}")
    print(f" UNDERSPECIFICATION — {label}")
    print(f" {context}")
    print(f"{'='*65}")
    print(f"  [{bar}] {t}/100  {tier}")

    for k,l in [("undeclared_preconditions", "Undeclared preconditions"),
            ("software_default_risk",    "Software default risk   "),
            ("implicit_data_coupling",   "Implicit data coupling  "),
            ("contract_completeness",    "Contract completeness   "),
            ("result_centrality",        "Result centrality       ")]:
        v = r.get(k,0)
        print(f"  {l}  {v:>2}  {'X'*v + '.'*(20-v)}")
    print(f"  Primary signal: {r.get('primary_signal','')}")
    print(f"  Load-bearing:   {', '.join(r.get('load_bearing_functions',[]))}")
    print(f"  Summary: {r.get('summary','')}")


def print_comparison(oA, oB, bA, bB, lA, lB):
    if not oA or not oB:
        print(f"  Cannot compare — missing results for {'' if oA else lA}{'' if oB else ' / '+lB}")
        return
    W = 72
    def conv(a,b,t=10): return "CONVERGE" if abs(a-b)<=t else "DIVERGE"
    def dlt(a,b): d=b-a; return f"+{d}" if d>0 else str(d) if d<0 else "0"
    tA = oA.get("total", 0)
    tB = oB.get("total", 0)
    sA = tA / 100.0
    sB = tB / 100.0
    print(f"\n{'='*W}")
    print(f" ENSEMBLE COMPARISON: {lA} vs {lB}")
    print(f"{'='*W}")
    for s,l in [(sA,lA),(sB,lB)]:
        bar = "X"*int(s*32)+"."*(32-int(s*32))
        print(f"  {l:<22} [{bar}] {int(s*100):>3}/100")
    print(f"  Spread: {abs(int(sA*100)-int(sB*100))} pts  Overall: {conv(int(sA*100),int(sB*100),15)}")
    print(f"\n{'-'*W}  DIMENSIONS")
    print(f"  {'Dimension':<26} {lA[:10]:>10}  {lB[:10]:>10}  Delta  Signal")
    # Use methods dimensions if running social science mode, else code dimensions
    dims = (
        [("undeclared_preconditions", "Undeclared preconditions"),
         ("software_default_risk",    "Software default risk   "),
         ("implicit_data_coupling",   "Implicit data coupling  "),
         ("contract_completeness",    "Contract completeness   "),
         ("result_centrality",        "Result centrality       ")]
    )
    for k,l in dims:
        vA,vB = oA.get(k,0),oB.get(k,0)
        print(f"  {l:<26} {vA:>10}  {vB:>10}  {dlt(vA,vB):>5}  {conv(vA,vB)}")
    psA = oA.get("primary_signal","")
    psB = oB.get("primary_signal","")
    print(f"\n{'-'*W}  PRIMARY SIGNAL")
    print(f"  {lA}: {psA}")
    print(f"  {lB}: {psB}")
    both_sg = "subgraph" in psA.lower() and "subgraph" in psB.lower()
    both_party = any(w in psA.lower() for w in ["party","democrat","coding","identification"]) and \
                 any(w in psB.lower() for w in ["party","democrat","coding","identification"])
    if both_sg:
        print("\n  CONVERGENCE: Both models name get_subgraphs")
        print("  Orphaned invariant confirmed at 2x confidence")
        print("  Same structural fact. Two training lineages. Same finding.")
    elif both_party:
        print("\n  CONVERGENCE: Both models flag party identification coding as primary signal")
        print("  Verified failure node (Kohler et al. 2026, Table 3) detected at 2x confidence")
        print("  Underspecification confirmed before any re-implementation attempt.")
    else:
        print("\n  DIVERGENCE on primary signal — composition seam located")
    fnsA = set(f.lower() for f in oA.get("load_bearing_functions",[]))
    fnsB = set(f.lower() for f in oB.get("load_bearing_functions",[]))
    conv_fns = fnsA & fnsB
    print(f"\n{'-'*W}  LOAD-BEARING FUNCTIONS")
    if conv_fns:
        print(f"  Convergent (both models): {', '.join(sorted(conv_fns))}  [2x confirmed]")
    if fnsA-fnsB: print(f"  {lA} only: {', '.join(sorted(fnsA-fnsB))}")
    if fnsB-fnsA: print(f"  {lB} only: {', '.join(sorted(fnsB-fnsA))}")
    print(f"\n{'-'*W}  IMPLEMENTER BURDEN")
    for b,l in [(bA,lA),(bB,lB)]:
        # Support both code (caller_must_know) and methods (must_know) schemas
        items = b.get("caller_must_know", b.get("must_know", []))
        stated_key = "stated_in_interface" if "caller_must_know" in b else "stated_in_methods"
        stated = all(i.get(stated_key, False) for i in items)
        print(f"  {l}: {'ALL STATED' if stated else 'UNSTATED items found'}")
        for item in items:
            m = "stated" if item.get(stated_key) else "UNSTATED"
            print(f"    [{m}] {item.get('item','')[:65]}")
    print(f"\n{'='*W}")
    print("  Agreement establishes confidence.")
    print("  Disagreement locates structure.")
    print("  The most valuable signal is where a model finds what another read past.")
    print("  That is where the orphaned invariants live.")
    print(f"{'='*W}")
user_prompt = SETTELE_METHODS
print("Prompts and helpers ready")

Prompts and helpers ready


### Ground Truth — Verified Failure Case

The Settele (2022) methods description above says **"ANES five-point scale"** but does not specify which numeric codes map to "Democrat".

**Kohler et al. (2026) Table 3** documents this as a verified failure case:
- One agent incorrectly assumed codes **A1/A2 = Democrat** → grade D
- One agent correctly assumed codes **A4/A5 = Democrat** → grade B

The paper itself provides no basis for choosing between these interpretations.
A detection methodology that flags this ambiguity *before* re-implementation
predicts the failure in advance.


In [ ]:
#@title 1.5 Run Apertus 8B
print("--- Apertus 8B ---")
start_server(APERTUS_8B_FILE, n_gpu_layers=-1, wait=30)
result_apertus_8b = run_prompt(METHODS_OPACITY_SYSTEM, "Apertus 8B methods opacity")
burden_apertus_8b = run_prompt(IMPLEMENTER_BURDEN_SYSTEM, "Apertus 8B implementer burden", 600)
print_result(result_apertus_8b, "Apertus 8B")

--- Apertus 8B ---
  Loading Apertus-8B-Instruct-2509-Q5_K_M.gguf (30s)...
  Server ready
  Running Apertus 8B methods opacity...
{'undeclared_preconditions': 15, 'software_default_risk': 10, 'implicit_data_coupling': 12, 'contract_completeness': 10, 'result_centrality': 18, 'total': 65, 'opacity_score': 0.65, 'primary_signal': 'The study relies on a self-reported party identification variable, which may introduce measurement error and bias. The ANES five-point scale is not fully specified, and the coding of party identification may affect the results.', 'ambiguity_zones': ['The ANES five-point scale for party identification is not fully specified.', 'The coding of party identification may affect the results.', 'The clustering of standard errors at the individual level is not fully specified.'], 'summary': 'The methods description is moderately underspecified, with significant gaps in the specification of party identification coding, software defaults, and data dependencies. The relian

In [ ]:
#@title 1.6 Run Mistral 7B
print("--- Mistral 7B ---")
start_server(MISTRAL_7B_FILE, n_gpu_layers=-1, wait=30)
result_mistral_7b = run_prompt(METHODS_OPACITY_SYSTEM, "Mistral 7B methods opacity")
burden_mistral_7b = run_prompt(IMPLEMENTER_BURDEN_SYSTEM, "Mistral 7B implementer burden", 600)
print_result(result_mistral_7b, "Mistral 7B")

--- Mistral 7B ---
  Loading Mistral-7B-Instruct-v0.3-Q5_K_M.gguf (30s)...
  Server ready
  Running Mistral 7B methods opacity...
{'undeclared_preconditions': 2, 'software_default_risk': 0, 'implicit_data_coupling': 2, 'contract_completeness': 18, 'result_centrality': 16, 'total': 48, 'opacity_score': 0.48, 'primary_signal': 'Implicit data coupling due to unspecified coding of the party identification variable', 'ambiguity_zones': ['Software used for data analysis', 'Data preprocessing steps', 'Specific details about the construction of the policy support index'], 'summary': "The study's methods description is moderately underspecified, with significant implicit data coupling and incomplete analytical contracts. The re-implementer must infer the coding of the party identification variable and guess the data preprocessing steps and construction of the policy support index."}
  Running Mistral 7B implementer burden...
{'undeclared_preconditions': 0, 'software_default_risk': 0, 'implicit_

In [ ]:
#@title 1.7 Ensemble comparison: Apertus 8B vs Mistral 7B
print_comparison(result_apertus_8b, result_mistral_7b,
                 burden_apertus_8b, burden_mistral_7b,
                 "Apertus 8B", "Mistral 7B")


 ENSEMBLE COMPARISON: Apertus 8B vs Mistral 7B
  Apertus 8B             [XXXXXXXXXXXXXXXXXXXX............]  65/100
  Mistral 7B             [XXXXXXXXXXXXXXX.................]  48/100
  Spread: 17 pts  Overall: DIVERGE

------------------------------------------------------------------------  DIMENSIONS
  Dimension                  Apertus 8B  Mistral 7B  Delta  Signal
  Undeclared preconditions           15           2    -13  DIVERGE
  Software default risk              10           0    -10  CONVERGE
  Implicit data coupling             12           2    -10  CONVERGE
  Contract completeness              10          18     +8  CONVERGE
  Result centrality                  18          16     -2  CONVERGE

------------------------------------------------------------------------  PRIMARY SIGNAL
  Apertus 8B: The study relies on a self-reported party identification variable, which may introduce measurement error and bias. The ANES five-point scale is not fully specified, and the coding 

---
## Section 2 — Depth of Read: Apertus 70B vs Mistral 7B
**Requires A100 — any paid Colab plan.**

Replaces Apertus 8B with 70B. Tests whether a much larger model finds structural
contracts that 8B-class models read past. This is the Appendix A reproduction.

Standard A100 (40GB): set `A100_70B_PARTIAL_LAYERS` in Section 0 if needed,
or enable High-RAM via Runtime > Change runtime type (Pro).


In [ ]:
#@title 2.1 Download Apertus 70B
if not can_run_s2:
    print("Section 2 requires A100 (any paid Colab plan).")
    print("Section 1 above fully demonstrates the methodology.")
else:
    quant = "5_K_S" if (is_high_ram and is_a100) else "4_K_M"
    APERTUS_70B_FILE = f"Apertus-70B-Instruct-2509-Q{quant}.gguf"
    APERTUS_70B_REPO = "unsloth/Apertus-70B-Instruct-2509-GGUF"
    print(f"Downloading Apertus 70B Q{quant}  (5-15 min)...")
    if not os.path.exists(f"/content/{APERTUS_70B_FILE}"):
        hf_hub_download(repo_id=APERTUS_70B_REPO,
                        filename=APERTUS_70B_FILE, local_dir="/content")
    print(f"  {APERTUS_70B_FILE}")

Apertus-70B-Instruct-2509-Q5_K_S.gguf:   0%|          | 0.00/48.7G [00:00<?, ?B/s]

  Apertus-70B-Instruct-2509-Q5_K_S.gguf


In [ ]:
#@title 2.2 Run Apertus 70B
if not can_run_s2:
    print("Skipping.")
else:
    print(f"--- Apertus 70B (layers: {layers_70b}) ---")
    start_server(APERTUS_70B_FILE, n_gpu_layers=layers_70b, wait=60)
    result_apertus_70b = run_prompt(METHODS_OPACITY_SYSTEM, "Apertus 70B opacity")
    burden_apertus_70b = run_prompt(IMPLEMENTER_BURDEN_SYSTEM, "Apertus 70B caller burden", 600)
    print_result(result_apertus_70b, "Apertus 70B")

--- Apertus 70B (layers: -1) ---
  Loading Apertus-70B-Instruct-2509-Q5_K_S.gguf (60s)...
  Server ready
  Running Apertus 70B opacity...
{'undeclared_preconditions': 12, 'software_default_risk': 10, 'implicit_data_coupling': 15, 'contract_completeness': 18, 'result_centrality': 15, 'total': 60, 'opacity_score': 0.6, 'primary_signal': "The exact coding scheme for the party identification variable (e.g., whether 'strong Democrat' is coded as 1 or 2 in the ANES scale) is not specified, which could affect the partisan gap analysis.", 'ambiguity_zones': ['Coding of the party identification variable (ANES scale)', 'Software defaults for the regression estimator (e.g., type of standard error clustering)', 'Data filtering rules for the policy support index (e.g., handling missing values or non-numeric responses)'], 'summary': 'The methods description is moderately underspecified, with key gaps in data coding, software defaults, and data preprocessing. The most critical underspecification is t

In [ ]:
#@title 2.3 Comparison: Apertus 70B vs Mistral 7B  (Appendix A reproduction)
if not can_run_s2:
    print("Skipping.")
else:
    print_comparison(result_apertus_70b, result_mistral_7b,
                     burden_apertus_70b, burden_mistral_7b,
                     "Apertus 70B", "Mistral 7B")
    # Appendix A check
    items_70 = burden_apertus_70b.get("caller_must_know",[])
    items_7  = burden_mistral_7b.get("caller_must_know",[])
    stated_70 = all(i.get("stated_in_interface",False) for i in items_70)
    stated_7  = all(i.get("stated_in_interface",False) for i in items_7)
    print()
    print("APPENDIX A CHECK:")
    print(f"  Apertus 70B caller burden: {'ALL STATED' if stated_70 else 'UNSTATED items found'}")
    print(f"  Mistral 7B  caller burden: {'ALL STATED' if stated_7  else 'UNSTATED items found'}")
    if not stated_70:
        print("  70B found deeper orphaned invariants — paper finding confirmed")


 ENSEMBLE COMPARISON: Apertus 70B vs Mistral 7B
  Apertus 70B            [XXXXXXXXXXXXXXXXXXX.............]  60/100
  Mistral 7B             [XXXXXXXXXXXXXXX.................]  48/100
  Spread: 12 pts  Overall: CONVERGE

------------------------------------------------------------------------  DIMENSIONS
  Dimension                  Apertus 70  Mistral 7B  Delta  Signal
  Undeclared preconditions           12           2    -10  CONVERGE
  Software default risk              10           0    -10  CONVERGE
  Implicit data coupling             15           2    -13  DIVERGE
  Contract completeness              18          18      0  CONVERGE
  Result centrality                  15          16     +1  CONVERGE

------------------------------------------------------------------------  PRIMARY SIGNAL
  Apertus 70B: The exact coding scheme for the party identification variable (e.g., whether 'strong Democrat' is coded as 1 or 2 in the ANES scale) is not specified, which could affect the par

---
## Section 3 — Full Architecture Differential: Apertus 70B vs Mistral Nemo 12B
**Requires A100 — any paid Colab plan.**

The most heterogeneous comparison in the notebook: Apertus 70B (Swiss, Llama-family)
vs Mistral Nemo 12B (French, independent architecture, different training corpus).
A finding confirmed at this diversity level approaches the paper's 3x confidence
threshold for confirmed orphaned invariants.


In [ ]:
#@title 3.1 Download Mistral Nemo 12B
if not can_run_s2:
    print("Section 3 requires A100 + High-RAM or 80GB A100.")
    print("Enable High-RAM: Runtime > Change runtime type > High-RAM  (Pro+)")
    print()
    print("What Section 3 adds:")
    print("  Mistral Nemo 12B is from a completely different training lineage")
    print("  than Apertus. Convergence between Apertus 70B and Mistral Nemo")
    print("  tests whether structural signals cross the Swiss/French architectural")
    print("  boundary — not just the 8B/70B size boundary.")
    print("  Convergence here approaches the paper's 3x confirmation threshold.")
else:
    NEMO_FILE = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
    NEMO_REPO = "bartowski/Mistral-Nemo-Instruct-2407-GGUF"
    print(f"Downloading Mistral Nemo 12B...")
    if not os.path.exists(f"/content/{NEMO_FILE}"):
        hf_hub_download(repo_id=NEMO_REPO, filename=NEMO_FILE, local_dir="/content")
    print(f"  {NEMO_FILE}")

Mistral-Nemo-Instruct-2407-Q5_K_M.gguf:   0%|          | 0.00/8.73G [00:00<?, ?B/s]

  Mistral-Nemo-Instruct-2407-Q5_K_M.gguf


In [ ]:
#@title 3.2 Run Mistral Nemo 12B
if not can_run_s2:
    print("Skipping.")
else:
    print("--- Mistral Nemo 12B ---")
    start_server(NEMO_FILE, n_gpu_layers=-1, wait=45)
    result_nemo = run_prompt(METHODS_OPACITY_SYSTEM, "Mistral Nemo 12B opacity")
    burden_nemo = run_prompt(IMPLEMENTER_BURDEN_SYSTEM, "Mistral Nemo 12B caller burden", 600)
    print_result(result_nemo, "Mistral Nemo 12B")

--- Mistral Nemo 12B ---
  Loading Mistral-Nemo-Instruct-2407-Q5_K_M.gguf (45s)...
  Server ready
  Running Mistral Nemo 12B opacity...
{'undeclared_preconditions': 12, 'software_default_risk': 8, 'implicit_data_coupling': 15, 'contract_completeness': 10, 'result_centrality': 13, 'total': 58, 'opacity_score': 0.58, 'primary_signal': 'The exact coding scheme for the party identification variable is not specified.', 'ambiguity_zones': ['The specific software environment and version used for the analysis are not declared.', 'The handling of missing data, if any, is not described.', 'The method used to construct the policy support index is not detailed.'], 'summary': 'The methods description is moderately underspecified, with key details about data handling, software environment, and index construction left implicit.'}
  Running Mistral Nemo 12B caller burden...
{'undeclared_preconditions': 0, 'software_default_risk': 0, 'implicit_data_coupling': 0, 'contract_completeness': 0, 'result_cent

In [ ]:
#@title 3.3 Comparison: Apertus 70B vs Mistral Nemo 12B
if not can_run_s2:
    print("Skipping.")
else:
    print_comparison(result_apertus_70b, result_nemo,
                     burden_apertus_70b, burden_nemo,
                     "Apertus 70B", "Mistral Nemo 12B")


 ENSEMBLE COMPARISON: Apertus 70B vs Mistral Nemo 12B
  Apertus 70B            [XXXXXXXXXXXXXXXXXXX.............]  60/100
  Mistral Nemo 12B       [XXXXXXXXXXXXXXXXXX..............]  57/100
  Spread: 3 pts  Overall: CONVERGE

------------------------------------------------------------------------  DIMENSIONS
  Dimension                  Apertus 70  Mistral Ne  Delta  Signal
  Undeclared preconditions           12          12      0  CONVERGE
  Software default risk              10           8     -2  CONVERGE
  Implicit data coupling             15          15      0  CONVERGE
  Contract completeness              18          10     -8  CONVERGE
  Result centrality                  15          13     -2  CONVERGE

------------------------------------------------------------------------  PRIMARY SIGNAL
  Apertus 70B: The exact coding scheme for the party identification variable (e.g., whether 'strong Democrat' is coded as 1 or 2 in the ANES scale) is not specified, which could affect t

In [ ]:
#@title 3.4 Four-model convergence summary (runs if all sections completed)
from collections import Counter

available = {"Apertus 8B": result_apertus_8b,
             "Mistral 7B":  result_mistral_7b}
if can_run_s2:
    available["Apertus 70B"] = result_apertus_70b
if can_run_s2:                                        # FIX 1: was can_run_s2
    available["Mistral Nemo 12B"] = result_nemo

W = 72
print(f"{'='*W}")
print(f" CROSS-MODEL CONVERGENCE SUMMARY  ({len(available)} models)")
print(f"{'='*W}")

# domain-aware convergence check
def is_match(ps):
    ps = ps.lower()
    return ("subgraph" in ps or
            "party" in ps or
            "coding" in ps or
            "democrat" in ps or
            "identification" in ps or
            "anes" in ps)

print("\n  Primary signals:")
match_count = 0
for lbl, res in available.items():
    ps = res.get("primary_signal", "")
    found = is_match(ps)
    if found: match_count += 1
    marker = "[MATCH]" if found else "[      ]"
    print(f"  {marker} {lbl:<22} {ps[:50]}")

print(f"\n  Convergent signal found in {match_count}/{len(available)} models")
if match_count >= 3:
    print(f"  Confirmed at {match_count}x confidence — meets or exceeds 3x threshold")
elif match_count == 2:
    print(f"  Confirmed at 2x confidence")
elif match_count == 4:
    print(f"  4x confirmation — all models independently located the same signal")

# Ambiguity zones — more relevant than load-bearing functions for methods descriptions
print("\n  Ambiguity zone counts across all models:")
zone_counter = Counter()
for res in available.values():
    for zone in res.get("ambiguity_zones", []):
        # normalize to a short key
        z = zone.lower()
        if "party" in z or "coding" in z or "democrat" in z or "anes" in z:
            zone_counter["party identification coding"] += 1
        elif "software" in z or "default" in z or "stata" in z or "python" in z:
            zone_counter["software defaults"] += 1
        elif "index" in z or "policy support" in z:
            zone_counter["policy support index construction"] += 1
        elif "cluster" in z or "standard error" in z:
            zone_counter["clustering / standard errors"] += 1
        elif "missing" in z or "filter" in z or "exclusion" in z:
            zone_counter["data filtering / missing values"] += 1
        elif "treatment" in z or "assignment" in z or "randomiz" in z:
            zone_counter["treatment assignment specification"] += 1
        else:
            zone_counter[zone[:45]] += 1

if zone_counter:
    for zone, count in zone_counter.most_common(8):
        bar = "X" * count
        print(f"  {bar:<6} {zone:<45} [{count}x]")
else:
    print("  (no ambiguity zones returned)")

# FIX 3: load-bearing functions with fallback
fn_counter = Counter()
for res in available.values():
    for fn in res.get("load_bearing_functions", []):
        fn_counter[fn.lower()] += 1

if fn_counter:
    print("\n  Load-bearing function counts across all models:")
    for fn, count in fn_counter.most_common(8):
        bar = "X" * count
        print(f"  {bar:<6} {fn:<35} [{count}x]")

print(f"\n{'='*W}")
n = len(available)
lineages = []
if "Apertus 8B" in available or "Apertus 70B" in available:
    lineages.append("Swiss (CSCS / Llama-family)")
if "Mistral 7B" in available or "Mistral Nemo 12B" in available:
    lineages.append("French (Mistral architecture)")
print(f"  {n} model{'s' if n>1 else ''}. {len(lineages)} training {'lineage' if len(lineages)==1 else 'lineages'}.")
if len(lineages) > 1:
    print(f"  {' vs '.join(lineages)}.")
if match_count >= 2:
    print(f"\n  The underspecification signal is not a model artifact.")
    print(f"  It is a structural property of the methods description.")
    print(f"  Detected before any re-implementation attempt.")
print(f"{'='*W}")

 CROSS-MODEL CONVERGENCE SUMMARY  (4 models)

  Primary signals:
  [MATCH] Apertus 8B             The study relies on a self-reported party identifi
  [MATCH] Mistral 7B             Implicit data coupling due to unspecified coding o
  [MATCH] Apertus 70B            The exact coding scheme for the party identificati
  [MATCH] Mistral Nemo 12B       The exact coding scheme for the party identificati

  Convergent signal found in 4/4 models
  Confirmed at 4x confidence — meets or exceeds 3x threshold

  Ambiguity zone counts across all models:
  XXX    party identification coding                   [3x]
  XXX    software defaults                             [3x]
  XXX    policy support index construction             [3x]
  X      clustering / standard errors                  [1x]
  X      Data preprocessing steps                      [1x]
  X      data filtering / missing values               [1x]

  4 models. 2 training lineages.
  Swiss (CSCS / Llama-family) vs French (Mistral architectu